# 06 Insights

> Change only `RAW` / `PROCESSED` paths if your folder location is different.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import re, json

BASE = Path(r"C:\CHANGE\THIS\TO\YOUR\PROJECT")
RAW = BASE / "data" / "raw"
PROCESSED = BASE / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)
arrivals = pd.read_csv(RAW / "C:\\Users\\yalamanchi rohitha\\Downloads\\track3_agritech_dataset_files\\track3_mandi_arrivals.csv")
master = pd.read_csv(RAW / "C:\\Users\\yalamanchi rohitha\\Downloads\\track3_agritech_dataset_files\\track3_mandi_master.csv")
transport = pd.read_csv(RAW / "C:\\Users\\yalamanchi rohitha\\Downloads\\track3_agritech_dataset_files\\track3_transport_logistics.csv")
master
with open(RAW / "C:\\Users\\yalamanchi rohitha\\Downloads\\track3_agritech_dataset_files\\track3_price_and_msp.json", encoding="utf-8") as f:
    prices = pd.DataFrame(json.load(f))

weather = pd.read_excel(RAW / "C:\\Users\yalamanchi rohitha\\Downloads\\track3_agritech_dataset_files\\track3_weather_sensors.xlsx")

## Business insights

In [3]:
# ==========================================
# 06 INSIGHTS - BUSINESS INSIGHTS
# ==========================================

from pathlib import Path
import pandas as pd
import numpy as np

# Data paths
P = Path("../data/processed")
A = P / "analytics"

# ------------------------------------------
# LOAD ANALYTICS DATA
# ------------------------------------------

mandi = pd.read_csv(
    A / "mandi_kpi.csv"
)

daily = pd.read_csv(
    A / "daily_arrivals.csv"
)

crop = pd.read_csv(
    A / "crop_kpi.csv"
)

# ------------------------------------------
# PRICE PRESSURE
# Mandis where modal price is below MSP
# ------------------------------------------

price_pressure = (
    mandi[
        mandi["avg_msp_gap_pct"] < 0
    ]
    .sort_values("avg_msp_gap_pct")
)

# ------------------------------------------
# SUPPLY TREND
# ------------------------------------------

daily["rolling_7d"] = pd.to_numeric(
    daily["rolling_7d"],
    errors="coerce"
)

daily["arrival_quantity_qtl"] = pd.to_numeric(
    daily["arrival_quantity_qtl"],
    errors="coerce"
)

supply_stock = daily[
    daily["rolling_7d"] > 0
].copy()

supply_stock["arrival_vs_rolling_pct"] = (
    (
        supply_stock["arrival_quantity_qtl"]
        - supply_stock["rolling_7d"]
    )
    / supply_stock["rolling_7d"]
) * 100

# ------------------------------------------
# SUPPLY SHOCKS
# Arrival is 30%+ below 7-day average
# ------------------------------------------

supply_shocks = (
    supply_stock[
        supply_stock["arrival_vs_rolling_pct"] <= -30
    ]
    .sort_values("arrival_vs_rolling_pct")
)

# ------------------------------------------
# DISPLAY INSIGHTS
# ------------------------------------------

print("========================================")
print("BUSINESS INSIGHTS")
print("========================================")

print(
    f"\nMandis below MSP: "
    f"{len(price_pressure)}"
)

print(
    f"Supply shock days: "
    f"{len(supply_shocks)}"
)

print("\nTop Price Pressure Mandis:")
display(
    price_pressure.head(10)
)

print("\nTop Supply Shock Days:")
display(
    supply_shocks.head(10)
)

print("\nCrop Performance:")
display(
    crop.sort_values(
        "total_arrival_qtl",
        ascending=False
    )
)

# ------------------------------------------
# SAVE INSIGHTS
# ------------------------------------------

insight_path = P / "insights"
insight_path.mkdir(
    parents=True,
    exist_ok=True
)

price_pressure.to_csv(
    insight_path / "price_pressure.csv",
    index=False
)

supply_shocks.to_csv(
    insight_path / "supply_shocks.csv",
    index=False
)

print("\nInsights saved successfully!")

BUSINESS INSIGHTS

Mandis below MSP: 43
Supply shock days: 54

Top Price Pressure Mandis:


,mandi_id,avg_modal_price,avg_msp,avg_msp_gap_pct
5,006,2420.337610,2090.000000,-4.851175
49,050,3994.130672,5048.750000,-4.790797
51,052,3484.696250,4239.600000,-4.095889
43,044,2825.830156,3078.888889,-4.059978
4,005,3089.585000,3841.777778,-3.543627
240,mandi013,3235.915333,3201.800000,-3.291751
32,033,4332.400511,4205.000000,-3.123669
35,036,4390.985000,3631.000000,-2.442151
91,M035,2768.652222,3031.142857,-2.396423
230,mandi003,3399.340643,3875.111111,-2.128404



Top Supply Shock Days:


,date,arrival_quantity_qtl,rolling_7d,arrival_vs_rolling_pct
27,2026-01-28,364.8410,2509.564014,-85.461977
246,2026-09-04,406.8900,2550.377771,-84.045893
162,2026-06-12,435.7868,2531.209529,-82.783456
80,2026-03-22,850.5542,2872.411214,-70.388843
44,2026-02-14,822.7500,2595.024171,-68.295093
244,2026-09-02,1034.6089,3077.636986,-66.383011
56,2026-02-26,1011.8900,2877.418714,-64.833411
196,2026-07-16,939.1800,2429.174000,-61.337475
224,2026-08-13,1385.3721,3550.814814,-60.984389
47,2026-02-17,1022.9800,2575.520043,-60.280643



Crop Performance:


,crop_name,total_arrival_qtl,avg_arrival_qtl
20,Sugarcane,178096.5537,234.955876
14,Mustard,177369.4257,242.307959
18,Sarso,171204.5316,240.793997
3,Cotton,168739.3032,229.890059
19,Sarson,165578.3762,234.862945
30,गन्ना,165107.5448,233.202747
29,कपास,160479.6036,227.953982
35,सरसों,159579.9316,238.179002
25,mustard,158742.8877,238.711109
6,Ganna,157588.0927,241.329392



Insights saved successfully!


## Weather impact — correlation, not causation

In [5]:
# ==========================================
# WEATHER IMPACT ANALYSIS
# ==========================================

from pathlib import Path
import pandas as pd
import numpy as np

P = Path("../data/processed")
A = P / "analytics"

# Load daily arrivals
daily = pd.read_csv(
    A / "daily_arrivals.csv",
    parse_dates=["date"]
)

# Load daily weather
weather = pd.read_csv(
    P / "weather_daily.csv",
    parse_dates=["date"]
)

# Merge arrivals + weather by date
weather_impact = daily.merge(
    weather,
    on="date",
    how="inner"
)

# Calculate correlation
weather_corr = (
    weather_impact[
        ["rainfall_mm", "arrival_quantity_qtl"]
    ]
    .corr()
    .iloc[0, 1]
)

print("========================================")
print("WEATHER IMPACT ANALYSIS")
print("========================================")

print(
    "Rainfall-arrival correlation:",
    round(weather_corr, 3)
)

print(
    "\nInterpretation: This measures association/correlation "
    "between rainfall and arrivals. It does NOT prove causation."
)

print("\nMerged records:", len(weather_impact))

# Save result
insight_path = P / "insights"
insight_path.mkdir(
    parents=True,
    exist_ok=True
)

weather_impact.to_csv(
    insight_path / "weather_impact.csv",
    index=False
)

print("\nWeather impact analysis saved successfully!")

WEATHER IMPACT ANALYSIS
Rainfall-arrival correlation: 0.005

Interpretation: This measures association/correlation between rainfall and arrivals. It does NOT prove causation.

Merged records: 151

Weather impact analysis saved successfully!


In [7]:
# ==========================================
# MANDI HEALTH SCORE
# ==========================================

from pathlib import Path
import pandas as pd
import numpy as np

P = Path("../data/processed")
A = P / "analytics"
out = P / "insights"
out.mkdir(parents=True, exist_ok=True)

# Load required data
mandi = pd.read_csv(A / "mandi_kpi.csv")
daily = pd.read_csv(A / "daily_arrivals.csv")
transport = pd.read_csv(P / "transport_fact.csv")

# ------------------------------------------
# PRICE SCORE
# Higher MSP gap = better
# ------------------------------------------

mandi["price_score"] = (
    mandi["avg_msp_gap_pct"]
    .rank(pct=True) * 100
)

# ------------------------------------------
# LOGISTICS SCORE
# Lower transit/delay = better
# ------------------------------------------

transport["transit_hours"] = pd.to_numeric(
    transport["transit_hours"],
    errors="coerce"
)

transport["delay"] = (
    transport["transit_hours"] > 24
)

logistics = (
    transport
    .groupby("mandi_id", as_index=False)
    .agg(
        avg_transit_hours=("transit_hours", "mean"),
        delay_rate=("delay", "mean")
    )
)

logistics["logistics_score"] = (
    1 - logistics["delay_rate"]
) * 100

# ------------------------------------------
# MERGE
# ------------------------------------------

mandi_performance = mandi.merge(
    logistics,
    on="mandi_id",
    how="left"
)

# ------------------------------------------
# FINAL HEALTH SCORE
# ------------------------------------------

mandi_performance["price_score"] = (
    mandi_performance["price_score"].fillna(50)
)

mandi_performance["logistics_score"] = (
    mandi_performance["logistics_score"].fillna(50)
)

mandi_performance["health_score"] = (
    mandi_performance["price_score"] * 0.60
    + mandi_performance["logistics_score"] * 0.40
)

# ------------------------------------------
# HEALTH CATEGORY
# ------------------------------------------

mandi_performance["health_category"] = pd.cut(
    mandi_performance["health_score"],
    bins=[-np.inf, 40, 70, np.inf],
    labels=[
        "Needs Attention",
        "Stable",
        "Healthy"
    ]
)

# ------------------------------------------
# SORT
# ------------------------------------------

mandi_performance = mandi_performance.sort_values(
    "health_score",
    ascending=False
)

# ------------------------------------------
# SAVE
# ------------------------------------------

mandi_performance.to_csv(
    out / "mandi_health_scores.csv",
    index=False
)

print("========================================")
print("MANDI HEALTH SCORE CREATED")
print("========================================")

display(
    mandi_performance[
        [
            "mandi_id",
            "avg_msp_gap_pct",
            "avg_transit_hours",
            "delay_rate",
            "health_score",
            "health_category"
        ]
    ].head(10)
)

print("\nInsights saved successfully!")

MANDI HEALTH SCORE CREATED


,mandi_id,avg_msp_gap_pct,avg_transit_hours,delay_rate,health_score,health_category
37,038,9.722240,10.016667,0.0,100.000000,Healthy
47,048,8.743591,13.916667,0.0,99.824561,Healthy
10,011,7.756697,17.128571,0.0,99.649123,Healthy
169,MANDI-056,6.913475,13.668421,0.0,99.473684,Healthy
159,MANDI-046,6.701469,12.180769,0.0,99.298246,Healthy
271,mandi044,6.332101,14.093750,0.0,99.122807,Healthy
236,mandi009,6.298478,8.021429,0.0,98.947368,Healthy
122,MANDI-009,6.195477,10.726087,0.0,98.771930,Healthy
78,M022,6.186777,13.804545,0.0,98.596491,Healthy
261,mandi034,6.176212,13.360000,0.0,98.421053,Healthy



Insights saved successfully!
